> **AI use declaration.** Parts of the code and text in this notebook were generated with the help of an AI assistant (Claude, by Anthropic), as allowed for this assignment. I (Kularathna G.G.A.S., 224110P) ran every cell, checked the outputs and the numbers quoted in the text, and made the final modelling decisions.


# 02 - Data Cleaning & Preprocessing

Getting the data ready for feature engineering:
1. **Cleaning** - fixing the problems found in notebook 01. I kept it simple: fix what is clearly wrong, leave real values alone, and mark anything I had to estimate with `is_imputed`.
2. **Preprocessing** - log-transforming the counts and encoding the districts.

**Input:** `dataset/processed/01_validated_weekly_data.csv`
**Output:** `dataset/processed/02_cleaned_weekly_data.csv`

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("../dataset")
IN_PATH = DATA_DIR / "processed" / "01_validated_weekly_data.csv"
OUT_PATH = DATA_DIR / "processed" / "02_cleaned_weekly_data.csv"

In [2]:
df = pd.read_csv(IN_PATH, parse_dates=["start_date", "end_date"])
df["is_imputed"] = False

print(df.shape)
df.head()

(26753, 7)


,year,week,start_date,end_date,district,cases,is_imputed
0,2006,52,2006-12-23,2006-12-29,Ampara,0,False
1,2006,52,2006-12-23,2006-12-29,Anuradhapura,4,False
2,2006,52,2006-12-23,2006-12-29,Badulla,2,False
3,2006,52,2006-12-23,2006-12-29,Batticaloa,0,False
4,2006,52,2006-12-23,2006-12-29,Colombo,71,False


## 1. Week dates

A few weeks aren't exactly 7 days apart (2009 w17 is 8 days and w22 is 6 days, and the 2026 weeks start on Mondays instead of Saturdays). The weeks are already in the right order, so I just number them 0, 1, 2, ... and set `start_date = first week + 7 * number`. This only moves some dates by 1-2 days.

In [3]:
week_no = df["start_date"].rank(method="dense").astype(int) - 1
new_start = df["start_date"].min() + pd.to_timedelta(week_no * 7, unit="D")

print("days moved:", (new_start - df["start_date"]).dt.days.value_counts().to_dict())

df["start_date"] = new_start
df["end_date"] = new_start + pd.Timedelta(days=6)

days moved: {0: 25662, -2: 961, -1: 130}


## 2. Missing Puttalam row

Puttalam has no row for 2026 week 7. It's a missing record, not 0 cases, so I fill it with the average of week 6 and week 8.

In [4]:
put = df[df["district"] == "Puttalam"].set_index(["year", "week"])["cases"]
fill = round((put[(2026, 6)] + put[(2026, 8)]) / 2)
print("week 6:", put[(2026, 6)], "| week 8:", put[(2026, 8)], "| fill:", fill)

date = df.loc[(df["year"] == 2026) & (df["week"] == 7), "start_date"].iloc[0]
new_row = pd.DataFrame([{
    "year": 2026, "week": 7, "start_date": date, "end_date": date + pd.Timedelta(days=6),
    "district": "Puttalam", "cases": fill, "is_imputed": True,
}])
df = pd.concat([df, new_row], ignore_index=True)
df = df.sort_values(["district", "start_date"]).reset_index(drop=True)

week 6: 24 | week 8: 26 | fill: 25


## 3. Broken weeks

These weeks are clearly wrong:
- **2026 w14, w24, w37:** the counts are attached to the wrong districts (e.g. Colombo suddenly ~10 cases, Kilinochchi and Mannar 20-100x their normal level for one week).
- **2022 w37:** the whole week collapsed to 53 cases nationally vs ~600 in the weeks around it.

For w14, w24 and 2022 w37 I remove the values and fill them in with a straight line between the week before and the week after.

2026 w37 is the last week in the data, so there is no week after it to fill from. I drop that week rather than make up numbers.

In [5]:
last_week = df["start_date"].max()
df = df[df["start_date"] != last_week].reset_index(drop=True)

bad = ((df["year"] == 2026) & df["week"].isin([14, 24])) | ((df["year"] == 2022) & (df["week"] == 37))
print("rows to fix:", bad.sum())

# before
df[bad & df["district"].isin(["Colombo", "Kilinochchi", "Mannar"])].pivot(index=["year", "week"], columns="district", values="cases")

rows to fix: 78


district   Colombo  Kilinochchi  Mannar
year week                              
2022 37         21            0       0
2026 14          9          117      47
     24         20          419     387

In [6]:
df.loc[bad, "cases"] = np.nan
df["cases"] = df.groupby("district")["cases"].transform(lambda s: s.interpolate()).round().astype(int)
df.loc[bad, "is_imputed"] = True

# after
df[bad & df["district"].isin(["Colombo", "Kilinochchi", "Mannar"])].pivot(index=["year", "week"], columns="district", values="cases")

district   Colombo  Kilinochchi  Mannar
year week                              
2022 37         82            2       0
2026 14        304            2       2
     24        822            5       5

## 4. Duplicated weeks

Some weeks have exactly the same 26 district counts as the week before, which can't really happen by chance - most likely the same report was copied twice. I can't tell which of the two is the real one, so I keep both and mark them with `dup_week`.

In [7]:
wide = df.pivot(index="start_date", columns="district", values="cases")
same_as_prev = wide.eq(wide.shift()).all(axis=1)
dup_dates = wide.index[same_as_prev | same_as_prev.shift(-1, fill_value=False)]

df["dup_week"] = df["start_date"].isin(dup_dates)
df[df["dup_week"]].drop_duplicates("start_date")[["year", "week", "start_date"]]

,year,week,start_date
264,2012,3,2012-01-14
265,2012,4,2012-01-21
781,2021,51,2021-12-11
782,2021,52,2021-12-18
945,2025,7,2025-02-01
946,2025,8,2025-02-08
1016,2026,25,2026-06-13
1017,2026,26,2026-06-20


4 pairs: 2012 w3/w4, 2021 w51/w52, 2025 w7/w8 and 2026 w25/w26.

## 5. Left as they are

- **Outbreak spikes:** these are real outbreaks, which is exactly what the model needs to learn. Not touched.
- **Zeros:** ~14% of rows, mostly small districts. Real values.
- **2026 w34-36:** a few districts drop a lot (e.g. Gampaha 567 -> 48). Probably late reports, but I can't prove it, so kept as reported.
- **`year`/`week`:** don't always match the calendar around new year, so I use `start_date` for ordering instead.

## 6. Preprocessing

The data is clean now, so the last step is to put it in a form that's easier to model:

- **Log of cases:** counts are very skewed (lots of zeros, outbreaks in the thousands), so I add `log_cases = log1p(cases)`. This stops a few huge outbreak weeks from dominating the training. `cases` itself is kept so the real numbers are still there.
- **District code:** models need numbers, so each district gets a code (alphabetical order).

No scaling - the models will be XGBoost / LightGBM, which are tree-based and don't care about scale.

In [8]:
df["log_cases"] = np.log1p(df["cases"])
df[["cases", "log_cases"]].describe().round(2)

,cases,log_cases
count,26728.00,26728.00
mean,33.63,2.35
std,86.86,1.54
min,0.00,0.00
25%,2.00,1.10
50%,9.00,2.30
75%,30.00,3.43
max,2631.00,7.88


In [9]:
df["district_code"] = df["district"].astype("category").cat.codes
df.drop_duplicates("district").set_index("district_code")["district"].sort_index().to_dict()

{0: 'Ampara',
 1: 'Anuradhapura',
 2: 'Badulla',
 3: 'Batticaloa',
 4: 'Colombo',
 5: 'Galle',
 6: 'Gampaha',
 7: 'Hambanthota',
 8: 'Jaffna',
 9: 'Kalmune',
 10: 'Kalutara',
 11: 'Kandy',
 12: 'Kegalle',
 13: 'Kilinochchi',
 14: 'Kurunegala',
 15: 'Mannar',
 16: 'Matale',
 17: 'Matara',
 18: 'Monaragala',
 19: 'Mullaitivu',
 20: 'NuwaraEliya',
 21: 'Polonnaruwa',
 22: 'Puttalam',
 23: 'Ratnapura',
 24: 'Trincomalee',
 25: 'Vavuniya'}

## Checks

In [10]:
assert df["cases"].notna().all() and (df["cases"] >= 0).all()
assert not df.duplicated(["district", "start_date"]).any()
assert (df.groupby("district").size() == df["start_date"].nunique()).all()
assert (df["start_date"].drop_duplicates().sort_values().diff().dropna().dt.days == 7).all()

print(df.shape)
print("imputed rows:", df["is_imputed"].sum(), "| duplicate-week rows:", df["dup_week"].sum())

(26728, 10)
imputed rows: 79 | duplicate-week rows: 208


## Save

In [11]:
df = df.sort_values(["start_date", "district"])
df.to_csv(OUT_PATH, index=False, date_format="%Y-%m-%d")
print("saved", len(df), "rows")

saved 26728 rows


## Summary

| Problem | What I did |
|---|---|
| Weeks not exactly 7 days apart | Re-spaced all weeks 7 days apart |
| Missing Puttalam row (2026 w7) | Filled with average of w6 and w8 |
| Wrong-district weeks (2026 w14, w24) and collapsed week (2022 w37) | Filled in between neighbouring weeks |
| 2026 w37 (wrong districts, last week) | Dropped |
| Duplicated weeks | Kept, marked `dup_week` |
| Spikes, zeros, low 2026 w34-36 counts | Kept as they are |
| Skewed counts | Added `log_cases` = log1p(cases) |
| District names | Added `district_code` |